In [ ]:
import pandas as pd

# =========================
# 1. 读取数据
# =========================
file_path = "Dataset_origin.xlsx"

# 默认读取第一个表单
df = pd.read_excel(file_path)

# =========================
# 2. 根据 DOC 和 UV254 生成 Code
# 相同 DOC + UV254 标记为同一个数字
# =========================
code_cols = ["DOC", "UV254"]

missing_cols = [col for col in code_cols + ["DBPs"] if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下列不存在：{missing_cols}")

df["Code"] = df.groupby(code_cols, dropna=False).ngroup() + 1

# =========================
# 实验数据统一编号为 0
# =========================
if "DOI" not in df.columns:
    raise ValueError("数据中不存在 DOI 列！")

df.loc[df["DOI"] == "This study", "Code"] = 0

# 将 Code 放到前面，或放到 DOI 后面
cols = df.columns.tolist()
cols.remove("Code")

if "Origin" in cols:
    insert_idx = cols.index("Origin") + 1
else:
    insert_idx = 0

cols.insert(insert_idx, "Code")
df = df[cols]


# =========================
# 3. 从每个 Code 中提取一行
# DBPs 优先级：TCM > BDCM > DCAA > TCAA
# 如果没有这些 DBPs，则取该 Code 下第一行
# =========================
priority_dbps = ["TCM", "BDCM", "DCAA", "TCAA"]

def select_one_sample(group):
    for dbp in priority_dbps:
        matched = group[group["DBPs"] == dbp]
        if not matched.empty:
            return matched.iloc[0]
    return group.iloc[0]

# 实验数据（全部保留）
experiment_df = df[df["Code"] == 0].copy()

# 文献数据（每个 Code 保留一条）
literature_df = (
    df[df["Code"] != 0]
    .groupby("Code", group_keys=False)
    .apply(select_one_sample)
    .reset_index(drop=True)
)

# 合并
df_in = pd.concat(
    [experiment_df, literature_df],
    ignore_index=True
)

# 按 Code 排序（Code=0 在最前）
df_in = df_in.sort_values(
    by="Code",
    kind="stable"
).reset_index(drop=True)


# =========================
# 4. 导出为 Data-Code.xlsx
# Lec 表单：完整数据 + Code
# In 表单：每个 Code 提取一行
# =========================
output_file = "Dataset-Code.xlsx"

with pd.ExcelWriter(output_file) as writer:
    df.to_excel(writer, sheet_name="O", index=False)
    df_in.to_excel(writer, sheet_name="IN", index=False)

print(f"✅ 已完成导出：{output_file}")
print("- Lec：完整数据，包含 Code 列")
print("- In：每个 Code 提取一行，优先 DBPs = TCM、BDCM、DCAA、TCAA")

In [1]:
import pandas as pd
import numpy as np
import warnings

from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore")


# =========================
# 1. 基本设置
# =========================
file_path = "Dataset-Code.xlsx"

sheet_O = "O"
sheet_IN = "IN"

protected_cols = ["Origin", "Code", "Type", "DOI"]

special_impute_cols = ["C1", "C2", "LMW", "MMW", "HMW"]

target_col = "C"

categorical_cols = ["DBPs"]

methods = ["RS", "Median", "KNN", "BR", "MICE", "MF"]

random_seed = 64


# =========================
# 2. 读取数据
# =========================
df_O = pd.read_excel(file_path, sheet_name=sheet_O)
df_IN = pd.read_excel(file_path, sheet_name=sheet_IN)


# =========================
# 3. 插值函数
# =========================
def impute_numeric_data(df, target_impute_cols, method, random_seed=64):
    df_imp = df.copy()

    target_impute_cols = [
        col for col in target_impute_cols
        if col in df_imp.columns
    ]

    if len(target_impute_cols) == 0:
        return df_imp

    # 数值参考列：排除保护列
    numeric_cols = df_imp.select_dtypes(include=[np.number]).columns.tolist()

    numeric_use_cols = [
        col for col in numeric_cols
        if col not in protected_cols
    ]

    # 分类变量 DBPs 独热编码
    cat_use_cols = [
        col for col in categorical_cols
        if col in df_imp.columns
    ]

    if len(cat_use_cols) > 0:
        cat_encoded = pd.get_dummies(
            df_imp[cat_use_cols].astype("category"),
            columns=cat_use_cols,
            dummy_na=True
        )
    else:
        cat_encoded = pd.DataFrame(index=df_imp.index)

    # 构造算法输入矩阵
    data_num = df_imp[numeric_use_cols].copy()

    data_all = pd.concat(
        [data_num, cat_encoded],
        axis=1
    )

    # 记录原始列名，最后只回填这些列
    original_numeric_cols = data_num.columns.tolist()

    # =========================
    # RS
    # =========================
    if method == "RS":
        rng = np.random.default_rng(random_seed)

        for col in target_impute_cols:
            missing_mask = df_imp[col].isna()
            observed = df_imp.loc[~missing_mask, col].dropna()

            if len(observed) == 0:
                continue

            q_low = observed.quantile(0.05)
            q_high = observed.quantile(0.95)

            observed_trimmed = observed[
                (observed >= q_low) & (observed <= q_high)
            ]

            if len(observed_trimmed) == 0:
                observed_trimmed = observed

            df_imp.loc[missing_mask, col] = rng.choice(
                observed_trimmed.values,
                size=missing_mask.sum(),
                replace=True
            )

        return df_imp

    # =========================
    # Median
    # =========================
    elif method == "Median":
        imputer = SimpleImputer(strategy="median")

        data_all.loc[:, original_numeric_cols] = imputer.fit_transform(
            data_all[original_numeric_cols]
        )

    # =========================
    # KNN
    # =========================
    elif method == "KNN":
        n_samples = len(data_all)

        k = int(np.sqrt(n_samples))
        k = max(3, k)
        k = min(k, 20)
        k = min(k, n_samples - 1)

        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(data_all)

        imputer = KNNImputer(
            n_neighbors=k,
            weights="distance",
            metric="nan_euclidean"
        )

        data_imputed_scaled = imputer.fit_transform(data_scaled)

        data_imputed = pd.DataFrame(
            scaler.inverse_transform(data_imputed_scaled),
            columns=data_all.columns,
            index=df_imp.index
        )

        data_all = data_imputed

        print(f"KNN n_neighbors = {k}")

    # =========================
    # MF: MissForest 近似
    # =========================
    elif method == "MF":
        imputer = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=500,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=2,
                max_features="sqrt",
                bootstrap=True,
                random_state=random_seed,
                n_jobs=-1
            ),
            max_iter=30,
            initial_strategy="median",
            imputation_order="ascending",
            skip_complete=True,
            random_state=random_seed,
            tol=1e-3
        )

        data_all.loc[:, :] = imputer.fit_transform(data_all)

    # =========================
    # BR
    # =========================
    elif method == "BR":
        imputer = IterativeImputer(
            estimator=BayesianRidge(
                max_iter=300,
                tol=1e-3,
                alpha_1=1e-6,
                alpha_2=1e-6,
                lambda_1=1e-6,
                lambda_2=1e-6
            ),
            max_iter=30,
            initial_strategy="median",
            imputation_order="ascending",
            skip_complete=True,
            random_state=random_seed,
            tol=1e-3
        )

        data_all.loc[:, :] = imputer.fit_transform(data_all)

    # =========================
    # MICE
    # =========================
    elif method == "MICE":
        imputer = IterativeImputer(
            estimator=BayesianRidge(
                max_iter=300,
                tol=1e-3,
                alpha_1=1e-6,
                alpha_2=1e-6,
                lambda_1=1e-6,
                lambda_2=1e-6
            ),
            max_iter=30,
            initial_strategy="median",
            imputation_order="random",
            sample_posterior=True,
            skip_complete=True,
            random_state=random_seed,
            tol=1e-3
        )

        data_all.loc[:, :] = imputer.fit_transform(data_all)

    else:
        raise ValueError(f"Unknown method: {method}")

    # 只把目标插值列写回原始 dataframe
    for col in target_impute_cols:
        df_imp[col] = data_all[col]

    return df_imp


# =========================
# 4. 分方法处理
# =========================
with pd.ExcelWriter("IN_ed.xlsx") as writer_in:

    for method in methods:

        print(f"\n正在处理插值方法：{method}")

        # Step 1: 在 IN 表单中对 C1/C2/LMW/MMW/HMW 插值
        IN_imputed = impute_numeric_data(
            df=df_IN,
            target_impute_cols=special_impute_cols,
            method=method,
            random_seed=random_seed
        )

        IN_imputed.to_excel(writer_in, sheet_name=method, index=False)

        # Step 2: 按 Code 提取插值后的五列
        code_map = (
            IN_imputed[IN_imputed["Code"] != 0]
            .groupby("Code")[special_impute_cols]
            .mean()
        )

        # Step 3: 回填到 O 表单
        O_filled = df_O.copy()

        for col in special_impute_cols:
            mask = (
                (O_filled["Code"] != 0) &
                (O_filled["Code"].isin(code_map.index))
            )

            O_filled.loc[mask, col] = O_filled.loc[mask, "Code"].map(code_map[col])

        # Step 4: 对 O 表单中其他数值列继续插值
        numeric_cols = O_filled.select_dtypes(include=[np.number]).columns.tolist()

        other_impute_cols = [
            col for col in numeric_cols
            if col not in protected_cols
            and col != target_col
        ]

        final_dataset = impute_numeric_data(
            df=O_filled,
            target_impute_cols=other_impute_cols,
            method=method,
            random_seed=random_seed
        )

        # Step 5: 不对目标变量 C 插值
        if target_col in final_dataset.columns:
            final_dataset = final_dataset.dropna(subset=[target_col])

        # Step 6: 导出最终数据集
        output_name = f"Dataset_imputed_{method}.xlsx"
        final_dataset.to_excel(output_name, index=False)

        print(f"完成导出：{output_name}")


print("\n✅ 所有插值方法处理完成。")
print("✅ IN 表单插值结果已导出至：IN_ed.xlsx")


正在处理插值方法：RS
完成导出：Dataset_imputed_RS.xlsx

正在处理插值方法：Median
完成导出：Dataset_imputed_Median.xlsx

正在处理插值方法：KNN
KNN n_neighbors = 20
KNN n_neighbors = 20
完成导出：Dataset_imputed_KNN.xlsx

正在处理插值方法：BR
完成导出：Dataset_imputed_BR.xlsx

正在处理插值方法：MICE
完成导出：Dataset_imputed_MICE.xlsx

正在处理插值方法：MF
完成导出：Dataset_imputed_MF.xlsx

✅ 所有插值方法处理完成。
✅ IN 表单插值结果已导出至：IN_ed.xlsx
